## Imports

In [16]:
# GIS and spatial data libraries
import rasterio
from rasterio.crs import CRS
from rasterio.plot import show
from rasterio.transform import xy
from rasterio.warp import reproject, Resampling, transform_bounds

import pyproj
from pyproj import Transformer

from affine import Affine
from osgeo import gdal
gdal.UseExceptions()
from shapely.geometry import box

# Data analysis and machine learning libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from itertools import combinations
from scipy.stats import skew
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Data access libraries
import ee
import requests

import json
import os
import pip
import platform
import time

from pathlib import Path
from IPython.display import Image, display

# Check for load errors
print(
    "All libraries imported successfully\n" 
    f"gdal v.{gdal.__version__}\n"
    f"pip v.{pip.__version__}\n"
    f"Python v{platform.python_version()}"
)

All libraries imported successfully
gdal v.3.13.3
pip v.25.2
Python v3.13.7


In [3]:
# Verify paths
print(
    f"Working directory: {Path.cwd()}\n" 
    f"Python executable: {os.sys.executable}\n"
    f"Datasets folder exists: {Path("Datasets").exists()}\n"
    f"Assets folder exists: {Path("Assets").exists()}\n"
)

Working directory: c:\Users\matil\Documents\GitHub\GIS-Project
Python executable: c:\Users\matil\AppData\Local\Programs\Python\Python313\python.exe
Datasets folder exists: True
Assets folder exists: True



## Band and Blender Exports

In [20]:
data = np.load(r"Data Export\export_inputs.npz")

elevation_crop   = data["elevation_crop"]
slope_gdal_array = data["slope_gdal_array"]
ndvi_crop        = [data["ndvi_2024"], data["ndvi_2025"], data["ndvi_2026"]]
sentinel_transform = Affine(*data["sentinel_transform"])
col0, row0 = int(data["col0"]), int(data["row0"])
dw_crop = data["dw_crop"]
sentinel_transform = Affine(*data["sentinel_transform"])
sentinel_crs = CRS.from_wkt(str(data["sentinel_crs"]))

### Multi-band Export to Blender

In [ ]:
# --- settings ---
step = 1          # 1 = every pixel (matches the TIF); 2 = thin to a quarter of the points
z_shift = 0.0     # 0.0 keeps true elevations; use elev.min() later if you want the lowest point at z=0

# --- thin every array with the same slice so they stay aligned ---
elev    = elevation_crop[::step, ::step]
slope   = slope_gdal_array[::step, ::step]
ndvi_24 = ndvi_crop[0][::step, ::step]
ndvi_25 = ndvi_crop[1][::step, ::step]
ndvi_26 = ndvi_crop[2][::step, ::step]
dw = dw_crop[::step, ::step]

# --- shared validity mask ---
valid = (
    np.isfinite(elev) & np.isfinite(slope)
    & np.isfinite(ndvi_24) & np.isfinite(ndvi_25) & np.isfinite(ndvi_26)
)
rows, cols = np.where(valid)

# --- pixel -> UTM (multiply by step to recover the original pixel positions) ---
crop_transform = sentinel_transform * Affine.translation(col0, row0)
x_utm, y_utm = rasterio.transform.xy(crop_transform, rows * step, cols * step, offset="center")
x_utm, y_utm = np.asarray(x_utm), np.asarray(y_utm)

# --- origin = upper-left corner of the crop, so the cloud lines up with the TIF ---
origin_x, origin_y = crop_transform * (0, 0)

# --- one dict drives both the header and the columns ---
blender_points = {
    "x_pos_bld":     x_utm - origin_x,
    "y_pos_bld":     y_utm - origin_y,
    "z_elev_bld":    elev[rows, cols] - z_shift,
    "slope_degrees": slope[rows, cols],
    "ndvi_2024":     ndvi_24[rows, cols],
    "ndvi_2025":     ndvi_25[rows, cols],
    "ndvi_2026":     ndvi_26[rows, cols],
}

# --- quick checks before writing ---
for name in ("x_pos_bld", "y_pos_bld", "z_elev_bld"):
    arr = blender_points[name]
    print(f"{name}: min={arr.min():.1f}  max={arr.max():.1f}  span={arr.max()-arr.min():.1f}")
print("points:", len(rows))

# --- write last ---
np.savetxt(
    r"Data Export\terrain_points.csv",
    np.column_stack(list(blender_points.values())),
    delimiter=",",
    header=",".join(blender_points),
    comments="",
    fmt="%.3f",
)

x_pos_bld: min=35.0  max=10405.0  span=10370.0
y_pos_bld: min=-24835.0  max=-25.0  span=24810.0
z_elev_bld: min=-0.0  max=1492.6  span=1492.7
points: 2318682


### Multi-band Export to QGIS

In [26]:
# Store bands for export into dictionary
clean_export_datasets = {
    "NDVI_2024": ndvi_crop[0],
    "NDVI_2025": ndvi_crop[1],
    "NDVI_2026": ndvi_crop[2],
    "DEM": elevation_crop,
    "Slope": slope_gdal_array,
    "DW_v1": dw_crop
}

# Verify clean export bands have matching shapes
for name, array in clean_export_datasets.items():
    print(f"Key: {name}, Array Shape: {array.shape}")

# Select width and height of export datasets
export_array_width, export_array_height = clean_export_datasets["NDVI_2024"].shape[1], clean_export_datasets["NDVI_2024"].shape[0]

Key: NDVI_2024, Array Shape: (2486, 1043)
Key: NDVI_2025, Array Shape: (2486, 1043)
Key: NDVI_2026, Array Shape: (2486, 1043)
Key: DEM, Array Shape: (2486, 1043)
Key: Slope, Array Shape: (2486, 1043)
Key: DW_v1, Array Shape: (2486, 1043)


In [23]:
# Specify export path and crop the grid transform to match the cropped datasets
output_path = r"Data Export\export_bands.tif"
crop_transform = sentinel_transform * Affine.translation(col0, row0)

# Export clean datasets to GeoTIFF
with rasterio.open(output_path, "w",
                   driver="GTiff",
                   height=export_array_height,
                   width=export_array_width,
                   count=len(clean_export_datasets),
                   dtype="float32",
                   crs=sentinel_crs,
                   transform=crop_transform,
                   nodata=np.nan) as dst:

    for band_number, (name, raster) in enumerate(clean_export_datasets.items(), start=1):
        dst.write(raster.astype("float32"), band_number)
        dst.set_band_description(band_number, name)